In [69]:
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ds = xr.open_dataset("../../NC/compare.nc") 

In [70]:
def get_soft(ds, mode, names, time):
    stacks = []
    for name in names:
        var = f"{mode}_{name}"
        da = ds[var].sel(time=time)
        stacks.append(da)
    
    soft = xr.concat(stacks, dim="class")
    soft = soft.assign_coords({"class": names})
    soft = soft.fillna(0)
    return soft.values

In [71]:
def get_T_sum(soft_region,soft_basin,C):
    T_sum = np.zeros((C, C))
    for i in range(soft_region.shape[1]):      # lat
        for j in range(soft_region.shape[2]):  # lon

            M1 = soft_region[:, i, j]   # region
            M2 = soft_basin[:,  i, j]  # basin

            if np.isnan(M1).any() or np.isnan(M2).any():
                continue

            T = np.zeros((C, C))

            diag = np.minimum(M1, M2)
            for a in range(C):
                T[a, a] = diag[a]

            L = M1 - diag    # losses
            G = M2 - diag    # gains

            G_sum = G.sum()


            if G_sum > 0:
                for a in range(C):
                    for b in range(C):
                        if a != b:
                            T[a, b] = L[a] * G[b] / G_sum

            T_sum += T
    return T_sum

In [72]:
def composite_operator_fast(soft1, soft2,C):

    s1 = soft1
    s2 = soft2

    # -------- 1. diagonal part --------
    diag = np.minimum(s1, s2)               # shape (C, lat, lon)
    diag_sum = diag.sum(axis=(1, 2))        # shape (C,)

    # -------- 2. off-diagonal part --------
    L = s1 - diag
    G = s2 - diag

    G_sum = G.sum(axis=0)
    mask = (G_sum == 0)
    G_sum_safe = G_sum.copy()
    G_sum_safe[mask] = 1

    # compute off-diagonal transitions
    T_off = L[:, None, :, :] * G[None, :, :, :] / G_sum_safe[None, None, :, :]

    # mask out invalid pixels
    T_off[:, :, mask] = 0

    # set diagonal to zero (we will overwrite later)
    for c in range(C):
        T_off[c, c, :, :] = 0

    # sum off-diagonal
    T_off_sum = T_off.sum(axis=(2, 3))

    # -------- 3. add diagonal and off-diagonal --------
    T_sum = T_off_sum.copy()
    np.fill_diagonal(T_sum, diag_sum)

    return T_sum

In [73]:
years = list(range(2010, 2101, 10))   # 2010,2020,...,2100
years.insert(0, 2005)
names = ["agri","forest","grassland"]
C = len(names)

year = "2050"
name = "agri"
time =f"{year}-01-01"

In [74]:
soft_region = get_soft(ds, "region", names, time)
soft_basin  = get_soft(ds, "basin",  names, time)

#T_sum = get_T_sum(soft_region,soft_basin,C)
T_sum = composite_operator_fast(soft_region,soft_basin,C)
df = pd.DataFrame(T_sum, index=names, columns=names)
print(df)

                  agri        forest     grassland
agri       4624.077148    423.449005    444.458649
forest      666.731262  14579.417969    638.825562
grassland   420.117462    279.874451  12304.269531
